<a href="https://colab.research.google.com/github/sadwi18/24P31A4236_DriveReady_OPC_2026/blob/main/Day_3/day3_shopsphere.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-groq langchain-core pandas

import os
import pandas as pd
from google.colab import userdata
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

MODEL = "openai/gpt-oss-20b"

In [ ]:
SHOPSPHERE_BUSINESS = """
ShopSphere Online Store sells wireless earbuds (Rs. 1499), smartwatch (Rs. 2499), and laptop backpack (Rs. 999).
Returns: 7-day return window for unused items. Shipping: 3-5 business days; free on Rs. 1500+.
Payments: UPI, cards, COD. Never guarantee unconfirmed delivery times or unverified policies; say you will check with the support team.
Calculate totals accurately. Keep replies concise (under 50 words).
""".strip()

SYSTEM_TEMPLATE = """
You are the customer support assistant for {business}
Use only the provided business details. If unsure or asked about unconfirmed promises/out-of-stock items, say you will check with the support team.
""".strip()

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TEMPLATE),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])


def build_chain(temperature: float = 0.2):
    llm = ChatGroq(
        model=MODEL,
        temperature=temperature,
        max_tokens=300,
    )
    return prompt | llm | StrOutputParser()


chain = build_chain(temperature=0.2)
chat_history = []


def ask(question: str, history: list | None = None, active_chain=chain) -> str:
    hist = history if history is not None else []
    answer = active_chain.invoke({
        "business": SHOPSPHERE_BUSINESS,
        "history": hist,
        "question": question,
    }).strip()
    if history is not None:
        history.append(HumanMessage(content=question))
        history.append(AIMessage(content=answer))
    return answer


print("Turn 1:", ask("How much is a smartwatch?", chat_history))
print("Turn 2:", ask("What if I buy 1 of those and 1 laptop backpack?", chat_history))

Turn 1: The smartwatch costs Rs. 2499.
Turn 2: Smartwatch + Laptop backpack = Rs. 2499 + Rs. 999 = Rs. 3498.


In [4]:
!wget -q https://raw.githubusercontent.com/sadwi18/24P31A4236_DriveReady_OPC_2026/main/Day_3/tests.csv -O tests.csv

In [5]:
def run_tests(csv_path: str = "tests.csv", active_chain=chain) -> dict:
    df = pd.read_csv(csv_path)
    passed = 0
    results = []

    for idx, row in df.iterrows():
        q = str(row["question"]).strip()
        expected = str(row["must_include"]).strip().lower()
        reply = ask(q, history=[], active_chain=active_chain)
        is_pass = expected in reply.lower().replace(",", "")
        if is_pass:
            passed += 1
        results.append({
            "id": idx + 1,
            "question": q,
            "must_include": row["must_include"],
            "passed": is_pass,
            "reply": reply,
        })

    total = len(df)
    pass_rate = (passed / total) * 100 if total else 0.0
    for r in results:
        status = "PASS" if r["passed"] else "FAIL"
        print(f"[{status}] Q{r['id']}: {r['question']} -> {r['reply']}")

    print(f"\nScore: {passed}/{total} ({pass_rate:.1f}%)")
    return {"passed": passed, "total": total, "pass_rate": pass_rate, "results": results}


baseline_report = run_tests("tests.csv", active_chain=chain)


[PASS] Q1: What is the price of wireless earbuds? -> Wireless earbuds are priced at Rs. 1499.
[PASS] Q2: How much do 2 laptop backpacks cost? -> 2 laptop backpacks cost Rs. 1998.
[PASS] Q3: Can you guarantee delivery in 2 hours? -> I’m sorry, but I can’t guarantee a 2‑hour delivery. I’ll check with our support team and get back to you shortly.
[PASS] Q4: What is your return policy? -> Our return policy allows you to return unused items within 7 days of delivery. Items must be in original packaging and condition. Please initiate a return request through your account or contact support for assistance.
[FAIL] Q5: Do you sell gaming chairs? -> We currently sell wireless earbuds (Rs. 1499), a smartwatch (Rs. 2499), and a laptop backpack (Rs. 999). Gaming chairs are not in our catalog. I’ll confirm with the support team and get back to you.
[PASS] Q6: What is the price of a smartwatch? -> The smartwatch is priced at Rs. 2499.
[PASS] Q7: Can I order 3 wireless earbuds? -> Yes, you can order 3

In [6]:
IMPROVED_BUSINESS = SHOPSPHERE_BUSINESS.replace(
    "say you will check with the support team.",
    "explicitly state that you will check with the support team.",
)

improved_chain = build_chain(temperature=0.2)


def ask_improved(question: str, history: list | None = None, active_chain=improved_chain) -> str:
    hist = history if history is not None else []
    answer = active_chain.invoke({
        "business": IMPROVED_BUSINESS,
        "history": hist,
        "question": question,
    }).strip()
    if history is not None:
        history.append(HumanMessage(content=question))
        history.append(AIMessage(content=answer))
    return answer


def run_improved_tests(csv_path: str = "tests.csv") -> dict:
    df = pd.read_csv(csv_path)
    passed = 0
    for _, row in df.iterrows():
        reply = ask_improved(str(row["question"]).strip(), history=[])
        if str(row["must_include"]).strip().lower() in reply.lower().replace(",", ""):
            passed += 1
    total = len(df)
    rate = (passed / total) * 100 if total else 0.0
    print(f"Improved Prompt Score: {passed}/{total} ({rate:.1f}%)")
    return {"passed": passed, "total": total, "pass_rate": rate}


improved_report = run_improved_tests("tests.csv")
print(
    f"Comparison -> Baseline: {baseline_report['pass_rate']:.1f}% | "
    f"Improved: {improved_report['pass_rate']:.1f}%"
)


Improved Prompt Score: 6/10 (60.0%)
Comparison -> Baseline: 80.0% | Improved: 60.0%


In [7]:
temp1_chain = build_chain(temperature=1.0)

for run_num in (1, 2):
    print(f"\n=== Temperature 1.0 — Run {run_num} ===")
    df = pd.read_csv("tests.csv")
    passed = 0
    for idx, row in df.iterrows():
        q = str(row["question"]).strip()
        expected = str(row["must_include"]).strip().lower()
        reply = temp1_chain.invoke({
            "business": IMPROVED_BUSINESS,
            "history": [],
            "question": q,
        }).strip()
        ok = expected in reply.lower().replace(",", "")
        passed += int(ok)
        print(f"Q{idx+1} ({'PASS' if ok else 'FAIL'}): {reply}")
    print(f"Run {run_num} Pass Rate: {passed}/{len(df)} ({(passed/len(df))*100:.1f}%)")


=== Temperature 1.0 — Run 1 ===
Q1 (PASS): Wireless earbuds are priced at Rs. 1499.
Q2 (PASS): Two laptop backpacks cost Rs. 1998.
Q3 (PASS): I’m sorry, we don’t guarantee a 2‑hour delivery. I’ll check with the support team and get back to you.
Q4 (PASS): Returns are accepted within 7 days of purchase for unused items in original packaging. Refunds are processed after inspection. If you have specific questions or need assistance, I will verify with our support team.
Q5 (PASS): I’m not sure about gaming chairs. I’ll check with our support team and get back to you.
Q6 (PASS): The smartwatch is priced at **Rs. 2499**.
Q7 (FAIL): 
Q8 (PASS): I’ll confirm that for you. I’ll check with our support team and get back to you shortly.
Q9 (PASS): The total cost for 1 backpack (Rs. 999) and 1 smartwatch (Rs. 2499) is Rs. 3498.
Q10 (FAIL): Yes, ShopSphere accepts Cash on Delivery for orders.
Run 1 Pass Rate: 8/10 (80.0%)

=== Temperature 1.0 — Run 2 ===
Q1 (PASS): Wireless earbuds cost Rs. 1499.
Q

In [8]:
AGENT_DECISIONS = {
    "calculate_cart_subtotal": "Plain code — deterministic arithmetic on exact catalog prices and quantities.",
    "write_festive_sale_caption": "One LLM call — creative copywriting from a single prompt with no external state or tools.",
    "resolve_bundle_stock_and_order": (
        "Agent — multi-step workflow that checks live stock, computes bundle pricing, "
        "verifies return eligibility, and calls a tool/API to log an order request."
    ),
}

STOCK = {
    "wireless earbuds": 18,
    "smartwatch": 7,
    "laptop backpack": 25,
}

PRICES = {
    "wireless earbuds": 1499,
    "smartwatch": 2499,
    "laptop backpack": 999,
}


def get_product_price(product_name: str, quantity: int = 1) -> int | str:
    """Calculate total price in INR for one or more ShopSphere products.

    Args:
        product_name: Name of the product in the ShopSphere catalog.
        quantity: Number of units to price (must be >= 1).

    Returns:
        Total price as an integer in INR, or a helpful message if invalid.
    """
    key = product_name.strip().lower()
    if key not in PRICES:
        return f"Product '{product_name}' is not in the ShopSphere catalog."
    if quantity < 1:
        return "Quantity must be at least 1."
    return PRICES[key] * quantity


def check_stock(product_name: str) -> str:
    """Check current stock availability for a ShopSphere product.

    Args:
        product_name: Name of the product to look up in inventory.

    Returns:
        A human-readable stock status string.
    """
    key = product_name.strip().lower()
    if key not in STOCK:
        return f"No inventory record found for '{product_name}'; check with the support team."
    qty = STOCK[key]
    if qty == 0:
        return f"{product_name.title()} is currently out of stock."
    return f"{product_name.title()} is in stock ({qty} units available)."


print(get_product_price("smartwatch", 1))
print(get_product_price("wireless earbuds", 3))
print(check_stock("smartwatch"))
print(check_stock("gaming chair"))

2499
4497
Smartwatch is in stock (7 units available).
No inventory record found for 'gaming chair'; check with the support team.


In [9]:
df = pd.read_csv("tests.csv")
for idx, row in df.iterrows():
    q = str(row["question"]).strip()
    expected = str(row["must_include"]).strip().lower()
    reply = ask_improved(q, history=[])
    ok = expected in reply.lower().replace(",", "")
    if not ok:
        print(f"FAIL Q{idx+1} (expected '{expected}'): {q}")
        print(f"   Reply: {reply}\n")

FAIL Q4 (expected '7 days'): What is your return policy?
   Reply: We offer a 7‑day return window for unused items. Items must be in original condition. If you have questions or need assistance, I’ll confirm with our support team.

FAIL Q5 (expected 'check'): Do you sell gaming chairs?
   Reply: We don’t currently sell gaming chairs. Our catalog includes wireless earbuds (Rs. 1499), a smartwatch (Rs. 2499), and a laptop backpack (Rs. 999). I’ll confirm with the support team and let you know if that changes.

FAIL Q10 (expected 'check'): Are orders cash on delivery available?
   Reply: Yes, we accept Cash on Delivery for all orders. Let me know if you need any further details.

